In [ ]:
import numpy as np
import plumed
import matplotlib.pyplot as plt
import os
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
from scipy.stats import entropy
from scipy.interpolate import interp1d
def get_cvname(filename):
    data = plumed.read_as_pandas(filename)

    return data.columns[1:]

def calculate_distribution(cv_name, colvar, folder):
    os.chdir(f"{folder}")
    try:
        os.mkdir("FES")
    except FileExistsError:
        pass
    for cv in cv_name:
        if cv[0] == 'd':
            sigma = 0.05
        else:
            sigma = 0.0628
        os.system(f"python /root/FES_from_Reweighting.py -f {colvar} -o FES/FES{cv}.dat --sigma {sigma} --temp 310 --cv {cv} --bias no")
def plot_distribution(cv_name, index1, index2, output_file=None):
    plt.figure(figsize=(10, 6))

    data1 = plumed.read_as_pandas(f"{index1}/FES/FES{cv_name}.dat")
    data2 = plumed.read_as_pandas(f"{index2}/FES/FES{cv_name}.dat")
    data1_cv = data1[cv_name].copy()#/np.pi*180
    data2_cv = data2[cv_name].copy()#/np.pi*180

    # data1_cv[data1_cv < 0] += 360
    # data2_cv[data2_cv < 0] += 360

    # 排序
    data1_sorted = data1_cv.argsort()
    data2_sorted = data2_cv.argsort()

    data1_cv = data1_cv[data1_sorted]
    data2_cv = data2_cv[data2_sorted]
    
    stats_results,prob1,prob2 = compare_distributions(data1, data2, cv_name)
    
    # prob1 = prob1.values[data1_sorted]
    # prob2 = prob2.values[data2_sorted]
    plt.plot(data1_cv, prob1, label=f'Distribution {index1}')
    plt.plot(data2_cv, prob2, label=f'Distribution {index2}')

    # 添加统计结果到图表
    plt.text(0.05, 0.95, 
             f"KL divergence: {stats_results['kl_divergence']:.2e}\n"
             f"Overlap index: {stats_results['overlap_index']:.3f}\n"
             f"JS distance: {stats_results['js_distance']:.3f}\n"
             f"Significant difference: {stats_results['significant_difference']}",
             transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', alpha=0.8),
             verticalalignment='top')
    plt.xlabel(cv_name)
    plt.ylabel('Probability Density')
    plt.title(f'Distribution of {cv_name}')
    plt.legend()
    if output_file:
        plt.savefig(output_file)
    plt.close()
    return stats_results['significant_difference']

def compare_distributions(data1, data2,cv_name):
    kb = 0.0083145
    T= 310
    kbT = kb * T  
    x1, f1 = data1[cv_name], data1['file.free']
    x2, f2 = data2[cv_name], data2['file.free']

    p1_raw = np.exp(-(f1 - np.min(f1)) / kbT)
    p2_raw = np.exp(-(f2 - np.min(f2)) / kbT)
    
    p1_pdf = p1_raw / np.sum(p1_raw)
    p2_pdf = p2_raw / np.sum(p2_raw)

    x_min = min(np.min(x1), np.min(x2))
    x_max = max(np.max(x1), np.max(x2))

    num_points = max(len(x1), len(x2))
    x_unified = np.linspace(x_min, x_max, num_points)

    interp_p1 = interp1d(x1, p1_pdf, bounds_error=False, fill_value=0)
    interp_p2 = interp1d(x2, p2_pdf, bounds_error=False, fill_value=0)

    prob1_new = interp_p1(x_unified)
    prob2_new = interp_p2(x_unified)

    prob1_new /= (np.sum(prob1_new) + 1e-12)
    prob2_new /= (np.sum(prob2_new) + 1e-12) 

    
    js_dist = jensenshannon(prob1_new, prob2_new)

    kl_div = entropy(prob1_new, prob2_new)

    overlap = np.sum(np.minimum(prob1_new, prob2_new))

    cdf1 = np.cumsum(prob1_new)
    cdf2 = np.cumsum(prob2_new)
    ks_stat = np.max(np.abs(cdf1 - cdf2))
    
    results = {
        'js_distance': js_dist,     
        'kl_divergence': kl_div,     
        'overlap_index': overlap,    
        'max_cdf_diff': ks_stat,     
        'significant_difference': js_dist > 0.2 
    }
    
    return results, prob1_new, prob2_new

def main():

    colvar_file = "/workdir/AgOR10_28/AgOR10/AgOR10_bound/COLVAR"  
    colvar_name = "COLVAR" 

    cv_name = get_cvname(colvar_file)
    print(cv_name)
    for i in ['AgOR10_bound','AgOR10_apo/cuts1']:
        calculate_distribution(cv_name, colvar_name,f"/workdir//AgOR10_28/AgOR10/{i}")
    significant_cvs = []
    os.system("rm -rf /workdir//AgOR10_28/AgOR10/open_close_difference_figures/*.png")
    for cv in cv_name:
        if 'phi' not in cv and 'psi' not in cv and 'omega' not in cv:
            significant_difference = plot_distribution(cv, '/workdir//AgOR10_28/AgOR10/AgOR10_bound', '/workdir//AgOR10_28/AgOR10/AgOR10_apo/cuts1', f"/workdir//AgOR10_28/AgOR10/open_close_difference_figures/{cv}.png")
            if significant_difference:
                print(significant_difference)
                significant_cvs.append(cv)
    print(f"significant_cvs: {significant_cvs}")
    print(f"number of significant_cvs: {len(significant_cvs)}")
    with open("/workdir/AgOR10_28/AgOR10/significant_cvs.txt", "w") as f:
        for cv in significant_cvs:
            f.write(cv + "\n")
    
def write_plumed_input():
    with open('/workdir/AgOR10_28/AgOR10/significant_cvs.txt', 'r') as f:
        cvs = f.readlines()
    cvs = [i.strip() for i in cvs]
    significant_cvs = []
    with open('/workdir//AgOR10_28/AgOR10/plumed_from_2structures.dat', 'r') as f:
        lines = f.readlines()
    for cv in cvs:
        if cv[0] == 'd':
            for line in lines:
                if line.startswith(f'{cv.split(".")[0]}:'):
                    significant_cvs.append(line)
                    break
            significant_cvs.append(f'e{cv.split(".")[0]}: CUSTOM ARG={cv} FUNC=e^-x PERIODIC=NO\n')
        else:
            # chain = cv.split('_')[0]
            chain = 'A'
            dihedral = cv.split('_')[1]
            res = cv.split('_')[2][-3:]
            print(cv)
            significant_cvs.append(f'{cv}: TORSION ATOMS=@{dihedral}-{chain}{res}\n')
            significant_cvs.append(f'cos_{cv}: CUSTOM ARG={cv} FUNC=cos(x) PERIODIC=NO\n')
            significant_cvs.append(f'sin_{cv}: CUSTOM ARG={cv} FUNC=sin(x) PERIODIC=NO\n')
    with open('/workdir//AgOR10_28/AgOR10/plumed_from_2trajs.dat', 'w') as f:
        f.write('MOLINFO STRUCTURE=molinfo.pdb\n')
        f.write('C_center_1_460: CENTER ATOMS=13617,13624,13643,13659,13669,13681,13700,13721,13733,13747,13766,13790,13806,13823,13840,13847,13863,13870,13887,13907,13924,13944,13958,13979,14000,14011,14026,14033,14055,14077,14099,14111,14128,14152,14171,14192,14203,14222,14238,14252,14271,14290,14309,14328,14347,14366,14383,14404,14411,14428,14445,14455,14471,14485,14504,14521,14538,14553,14564,14576,14588,14604,14616,14628,14647,14661,14671,14685,14699,14718,14732,14749,14768,14788,14808,14827,14846,14858,14877,14893,14915,14932,14951,14972,14994,15006,15022,15046,15057,15079,15098,15118,15139,15161,15175,15194,15204,15223,15247,15261,15277,15289,15303,15314,15333,15345,15364,15384,15394,15409,15420,15434,15444,15468,15488,15505,15515,15534,15544,15563,15577,15599,15616,15640,15664,15683,15702,15722,15733,15749,15759,15766,15776,15790,15809,15829,15840,15856,15875,15886,15910,15924,15931,15950,15964,15984,16003,16018,16030,16041,16057,16079,16103,16122,16136,16150,16162,16177,16191,16205,16220,16234,16248,16267,16288,16300,16321,16333,16357,16376,16393,16412,16436,16450,16470,16493,16505,16525,16539,16549,16566,16577,16584,16594,16601,16618,16634,16654,16664,16683,16702,16723,16740,16760,16781,16802,16821,16837,16856,16867,16884,16894,16910,16921,16935,16946,16965,16977,16993,17012,17032,17043,17054,17078,17097,17116,17136,17146,17157,17172,17189,17208,17225,17242,17261,17283,17293,17312,17329,17353,17365,17384,17401,17416,17435,17446,17456,17470,17489,17501,17515,17531,17549,17561,17575,17586,17593,17608,17627,17647,17669,17679,17686,17697,17707,17719,17736,17755,17779,17794,17805,17822,17829,17845,17864,17876,17887,17894,17908,17915,17927,17941,17957,17976,17988,18004,18016,18035,18059,18066,18085,18106,18117,18131,18155,18172,18184,18204,18218,18228,18242,18262,18288,18300,18314,18324,18331,18345,18359,18379,18393,18400,18407,18423,18432,18444,18458,18465,18484,18498,18520,18542,18559,18574,18591,18610,18626,18650,18661,18671,18690,18712,18733,18757,18773,18788,18812,18829,18851,18868,18884,18900,18924,18943,18959,18973,18983,18999,19006,19018,19028,19049,19056,19072,19082,19101,19120,19139,19156,19173,19192,19206,19220,19234,19253,19267,19286,19300,19319,19338,19348,19369,19386,19396,19410,19432,19448,19462,19469,19485,19499,19515,19536,19546,19556,19570,19586,19605,19612,19633,19652,19671,19692,19706,19725,19732,19749,19765,19785,19804,19824,19835,19854,19874,19881,19895,19919,19938,19957,19972,19987,19998,20009,20020,20036,20053,20068,20078,20088,20109,20120,20131,20148,20172,20193,20205,20212,20223,20238,20253,20263,20285,20299,20319,20335,20352,20371,20387,20398,20415,20432,20443,20460,20482,20492,20509,20520,20539,20550,20557,20567,20589,20609,20629,20643,20659,20670,20689,20701,20720,20740,20750,20761,20777\n')
        f.write('D_center_1_460: CENTER ATOMS=21029,21036,21055,21071,21081,21093,21112,21133,21145,21159,21178,21202,21218,21235,21252,21259,21275,21282,21299,21319,21336,21356,21370,21391,21412,21423,21438,21445,21467,21489,21511,21523,21540,21564,21583,21604,21615,21634,21650,21664,21683,21702,21721,21740,21759,21778,21795,21816,21823,21840,21857,21867,21883,21897,21916,21933,21950,21965,21976,21988,22000,22016,22028,22040,22059,22073,22083,22097,22111,22130,22144,22161,22180,22200,22220,22239,22258,22270,22289,22305,22327,22344,22363,22384,22406,22418,22434,22458,22469,22491,22510,22530,22551,22573,22587,22606,22616,22635,22659,22673,22689,22701,22715,22726,22745,22757,22776,22796,22806,22821,22832,22846,22856,22880,22900,22917,22927,22946,22956,22975,22989,23011,23028,23052,23076,23095,23114,23134,23145,23161,23171,23178,23188,23202,23221,23241,23252,23268,23287,23298,23322,23336,23343,23362,23376,23396,23415,23430,23442,23453,23469,23491,23515,23534,23548,23562,23574,23589,23603,23617,23632,23646,23660,23679,23700,23712,23733,23745,23769,23788,23805,23824,23848,23862,23882,23905,23917,23937,23951,23961,23978,23989,23996,24006,24013,24030,24046,24066,24076,24095,24114,24135,24152,24172,24193,24214,24233,24249,24268,24279,24296,24306,24322,24333,24347,24358,24377,24389,24405,24424,24444,24455,24466,24490,24509,24528,24548,24558,24569,24584,24601,24620,24637,24654,24673,24695,24705,24724,24741,24765,24777,24796,24813,24828,24847,24858,24868,24882,24901,24913,24927,24943,24961,24973,24987,24998,25005,25020,25039,25059,25081,25091,25098,25109,25119,25131,25148,25167,25191,25206,25217,25234,25241,25257,25276,25288,25299,25306,25320,25327,25339,25353,25369,25388,25400,25416,25428,25447,25471,25478,25497,25518,25529,25543,25567,25584,25596,25616,25630,25640,25654,25674,25700,25712,25726,25736,25743,25757,25771,25791,25805,25812,25819,25835,25844,25856,25870,25877,25896,25910,25932,25954,25971,25986,26003,26022,26038,26062,26073,26083,26102,26124,26145,26169,26185,26200,26224,26241,26263,26280,26296,26312,26336,26355,26371,26385,26395,26411,26418,26430,26440,26461,26468,26484,26494,26513,26532,26551,26568,26585,26604,26618,26632,26646,26665,26679,26698,26712,26731,26750,26760,26781,26798,26808,26822,26844,26860,26874,26881,26897,26911,26927,26948,26958,26968,26982,26998,27017,27024,27045,27064,27083,27104,27118,27137,27144,27161,27177,27197,27216,27236,27247,27266,27286,27293,27307,27331,27350,27369,27384,27399,27410,27421,27432,27448,27465,27480,27490,27500,27521,27532,27543,27560,27584,27605,27617,27624,27635,27650,27665,27675,27697,27711,27731,27747,27764,27783,27799,27810,27827,27844,27855,27872,27894,27904,27921,27932,27951,27962,27969,27979,28001,28021,28041,28055,28071,28082,28101,28113,28132,28152,28162,28173,28189\n')
        f.write('B_center_1_460: CENTER ATOMS=6205,6212,6231,6247,6257,6269,6288,6309,6321,6335,6354,6378,6394,6411,6428,6435,6451,6458,6475,6495,6512,6532,6546,6567,6588,6599,6614,6621,6643,6665,6687,6699,6716,6740,6759,6780,6791,6810,6826,6840,6859,6878,6897,6916,6935,6954,6971,6992,6999,7016,7033,7043,7059,7073,7092,7109,7126,7141,7152,7164,7176,7192,7204,7216,7235,7249,7259,7273,7287,7306,7320,7337,7356,7376,7396,7415,7434,7446,7465,7481,7503,7520,7539,7560,7582,7594,7610,7634,7645,7667,7686,7706,7727,7749,7763,7782,7792,7811,7835,7849,7865,7877,7891,7902,7921,7933,7952,7972,7982,7997,8008,8022,8032,8056,8076,8093,8103,8122,8132,8151,8165,8187,8204,8228,8252,8271,8290,8310,8321,8337,8347,8354,8364,8378,8397,8417,8428,8444,8463,8474,8498,8512,8519,8538,8552,8572,8591,8606,8618,8629,8645,8667,8691,8710,8724,8738,8750,8765,8779,8793,8808,8822,8836,8855,8876,8888,8909,8921,8945,8964,8981,9000,9024,9038,9058,9081,9093,9113,9127,9137,9154,9165,9172,9182,9189,9206,9222,9242,9252,9271,9290,9311,9328,9348,9369,9390,9409,9425,9444,9455,9472,9482,9498,9509,9523,9534,9553,9565,9581,9600,9620,9631,9642,9666,9685,9704,9724,9734,9745,9760,9777,9796,9813,9830,9849,9871,9881,9900,9917,9941,9953,9972,9989,10004,10023,10034,10044,10058,10077,10089,10103,10119,10137,10149,10163,10174,10181,10196,10215,10235,10257,10267,10274,10285,10295,10307,10324,10343,10367,10382,10393,10410,10417,10433,10452,10464,10475,10482,10496,10503,10515,10529,10545,10564,10576,10592,10604,10623,10647,10654,10673,10694,10705,10719,10743,10760,10772,10792,10806,10816,10830,10850,10876,10888,10902,10912,10919,10933,10947,10967,10981,10988,10995,11011,11020,11032,11046,11053,11072,11086,11108,11130,11147,11162,11179,11198,11214,11238,11249,11259,11278,11300,11321,11345,11361,11376,11400,11417,11439,11456,11472,11488,11512,11531,11547,11561,11571,11587,11594,11606,11616,11637,11644,11660,11670,11689,11708,11727,11744,11761,11780,11794,11808,11822,11841,11855,11874,11888,11907,11926,11936,11957,11974,11984,11998,12020,12036,12050,12057,12073,12087,12103,12124,12134,12144,12158,12174,12193,12200,12221,12240,12259,12280,12294,12313,12320,12337,12353,12373,12392,12412,12423,12442,12462,12469,12483,12507,12526,12545,12560,12575,12586,12597,12608,12624,12641,12656,12666,12676,12697,12708,12719,12736,12760,12781,12793,12800,12811,12826,12841,12851,12873,12887,12907,12923,12940,12959,12975,12986,13003,13020,13031,13048,13070,13080,13097,13108,13127,13138,13145,13155,13177,13197,13217,13231,13247,13258,13277,13289,13308,13328,13338,13349,13365\n')

        f.write('\n')
        for line in significant_cvs:
            f.write(line)
        f.write('\nPRINT ARG=* STRIDE=1 FILE=COLVAR\n')

In [10]:
main()

+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


Index(['d_A_GLU145_A_VAL373.min', 'd_A_SER365_A_TRP52.min',
       'd_A_LEU48_A_VAL373.min', 'd_A_SER51_A_VAL373.min',
       'd_A_GLY156_A_LEU369.min', 'd_A_PHE160_A_VAL373.min',
       'd_A_ARG371_A_TRP52.min', 'd_A_GLU145_A_LEU369.min',
       'd_A_CYS49_A_LEU369.min', 'd_A_ASN375_A_GLU145.min',
       ...
       'A_chi1_VAL373', 'A_omega_TYR374', 'A_phi_TYR374', 'A_psi_TYR374',
       'A_chi1_TYR374', 'A_chi2_TYR374', 'A_omega_ASN375', 'A_phi_ASN375',
       'A_chi1_ASN375', 'A_chi2_ASN375'],
      dtype='object', length=377)
True
True
True


+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


True
True
True
True


+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


...

+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


True
True
True


+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


True
True
True
significant_cvs: ['d_A_GLU145_A_VAL373.min', 'd_A_SER365_A_TRP52.min', 'd_A_LEU48_A_VAL373.min', 'd_A_SER51_A_VAL373.min', 'd_A_GLY156_A_LEU369.min', 'd_A_PHE160_A_VAL373.min', 'd_A_ARG371_A_TRP52.min', 'd_A_GLU145_A_LEU369.min', 'd_A_CYS49_A_LEU369.min', 'd_A_ASN375_A_GLU145.min', 'd_A_ARG372_A_GLU145.min', 'd_A_LEU369_A_PHE45.min', 'd_A_LEU369_A_PHE42.min', 'd_A_THR368_A_TRP52.min', 'd_A_LEU369_A_THR46.min', 'd_A_LEU157_A_VAL373.min', 'd_A_GLN278_A_ILE141.min', 'd_A_ASN375_A_PHE45.min', 'd_A_ARG372_A_LEU58.min', 'd_A_ASN375_A_ASN54.min', 'd_A_ASN375_A_PHE142.min', 'd_A_ILE35_A_VAL373.min', 'd_A_ILE55_A_VAL373.min', 'd_A_THR41_A_VAL373.min', 'd_A_GLY147_A_VAL373.min', 'd_A_ALA62_A_ASN375.min', 'd_A_LEU369_A_TYR139.min', 'd_A_ARG372_A_VAL39.min', 'd_A_LEU370_A_VAL39.min', 'd_A_GLU145_A_GLU274.min', 'd_A_ARG372_A_PHE45.min', 'd_A_ARG161_A_LEU369.min', 'd_A_GLN278_A_TRP52.min', 'd_A_GLY147_A_TYR374.min', 'd_A_PHE142_A_VAL272.min', 'd_A_ALA28_A_VAL373.min', 'd_A_ASN275_A_IL

+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++
+++ Loading the PLUMED kernel runtime +++
+++ PLUMED_KERNEL="/usr/local/lib/libplumedKernel.so" +++


In [11]:
write_plumed_input()

A_chi1_VAL248
A_chi2_TYR250
A_chi1_ILE251
A_chi1_CYS252
A_chi1_GLU255
A_chi2_GLU255
A_chi3_GLU255
A_chi1_PHE256
A_chi2_PHE256
A_chi1_LEU257
A_chi2_LEU257
A_chi2_PHE259
A_chi1_MET262
A_chi2_MET262
A_chi3_MET262
A_chi1_LEU263
A_chi2_LEU263
A_chi1_PHE268
A_chi2_PHE268
A_chi1_ASN271
A_chi2_ASN271
A_chi3_GLN278
A_chi1_ILE279
A_chi1_ILE281
A_chi1_MET288
A_chi2_MET288
A_chi3_MET288
A_chi1_GLN292
A_chi2_GLN292
A_chi3_GLN292
A_chi1_PHE294
A_chi2_PHE294
A_chi1_TYR297
A_chi2_TRP298
A_chi2_GLU302
A_chi4_ARG304
A_chi2_GLU342
A_chi1_ILE343
A_chi1_VAL345
A_chi3_MET351
A_chi1_THR352
A_chi3_MET355
A_chi1_GLN357
A_chi3_GLN357
A_chi1_LEU359
A_chi2_TYR364
A_chi1_TYR366
A_chi2_TYR366
A_chi1_PHE367
A_chi1_THR368
A_chi2_LEU369
A_chi2_ARG371
A_chi4_ARG371
A_chi2_ARG372
A_chi3_ARG372
A_chi4_ARG372
A_chi1_TYR374
A_chi2_TYR374
A_chi2_ASN375


In [ ]:
# Small runner to call generate_pml_from_significant_cvs from cal_distribution.ipynb
# It reimplements the minimal parsing + generation logic so we can run it as a script
import re
import sys

significant_cvs_path = '/workdir/AgOR10_28/AgOR10/significant_cvs.txt'
mol_pdb_path = '/data/AgOR10_28/AgOR10/molinfo.pdb'
output_pml_path = '/workdir/AgOR10_28/AgOR10/vis_significant.pml'

def parse_entities(line):
    entities = []
    m_center = re.findall(r'([A-Z])[_-](?:center|centroid)[_\-]?(\d+)[_\-](\d+)', line)
    for ch, s, e in m_center:
        entities.append({'type': 'center', 'chain': ch, 'start': int(s), 'end': int(e)})
    m_res = re.findall(r'([A-Z])_([A-Z]{3})(\d+)', line)
    for ch, resname, rnum in m_res:
        entities.append({'type': 'res', 'chain': ch, 'resnum': int(rnum)})
    if not entities:
        m_simple = re.findall(r'([A-Z])[_]?(\d{1,4})', line)
        for ch, rnum in m_simple:
            entities.append({'type': 'res', 'chain': ch, 'resnum': int(rnum)})
    return entities[:2]

with open(significant_cvs_path, 'r') as f:
    lines = [ln.strip() for ln in f if ln.strip()]

distances = []
for ln in lines:
    ents = parse_entities(ln)
    if len(ents) < 2:
        continue
    a, b = ents[0], ents[1]
    safe_label = re.sub(r'[^0-9A-Za-z]', '_', ln)
    distances.append((safe_label, a, b))

with open(output_pml_path, 'w') as out:
    out.write(f"# PyMOL script generated from {significant_cvs_path}\n")
    out.write(f"load {mol_pdb_path}, mol\n")
    out.write("hide everything, mol\n")
    out.write("show cartoon, mol\n")
    out.write("bg_color white\n")
    out.write("set dash_width, 4\n")
    out.write("# Create distances (direct CA selections; residue ranges used for centroid-like entities)\n")
    out.write("pseudoatom cB, selection=chain B and resi 1-460\n")
    out.write("pseudoatom cC, selection=chain C and resi 1-460\n")
    out.write("pseudoatom cD, selection=chain D and resi 1-460\n")
    out.write("as spheres, cB or cC or cD\n")
    out.write("color grey, cB or cC or cD\n")
    out.write("set sphere_scale, 0.1\n")
    out.write("\n")
    out.write("util.cbc()\n")
    out.write("sele chainA, chain A\n")
    out.write("color purple, chainA\n")
    for label, entA, entB in distances:
        def sel(ent):
            if ent['type'] == 'res':
                return f"(mol and chain {ent['chain']} and resi {ent['resnum']} and name CA)"
            elif ent['type'] == 'center':
                return f"(c{ent['chain']})"
            else:
                return "(mol)"
        selA = sel(entA)
        selB = sel(entB)
        out.write(f"distance {label}, {selA}, {selB}\n")
        out.write(f"color red, {label}\n")
        out.write(f"hide labels, {label}\n")
    out.write("zoom mol\n")

print('Wrote', output_pml_path)
